# # 03 - NHANES Data Preparation

> **This notebook is part of the SleepHealth-AI project.**
> It is executed against the *production* pipeline in `src/`, so every number
> below is the same number reported in `REPORT.md`, `results/metrics/metrics.json`
> and the Streamlit app. Each final cell **asserts** that agreement, so this
> notebook cannot silently drift from the shipped system.

**Setup:** run this notebook from the `notebooks/` folder. The first cell moves
the working directory to the repository root and puts `src/` on `sys.path`.


## Why this notebook exists

This is the data-engineering stage: six separate CDC questionnaire files are merged into one analysis-ready table with **five prediction targets**.

* **Source:** NHANES 2017-March 2020 *pre-pandemic* release (`P_*.XPT`, May 2021).
* **Why this cycle:** the 2019-2020 cycle was suspended in March 2020 and was never released standalone, so NCHS published this combined pre-pandemic file instead. It roughly doubles the cohort versus the older 2017-2018 files.
* **Anti-leakage rule:** any questionnaire item used to *build* a label is excluded from that label's feature list.

In [1]:
# --- Reproducibility setup: run from notebooks/, work from the repo root ------
import os, sys, json
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":          # launched inside notebooks/
    ROOT = ROOT.parent
os.chdir(ROOT)                        # src/ scripts use repo-root-relative paths
sys.path.insert(0, str(ROOT / "src")) # so `import train_models` works

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Notebooks need the INLINE backend so plt.show() embeds the figure in the
# output. (The src/ scripts use "Agg" because they save PNGs to disk instead;
# copying that here would make every plot render as nothing.)
%matplotlib inline

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)

print("repo root :", ROOT)
print("python    :", sys.version.split()[0])
print("pandas    :", pd.__version__)


repo root : C:\Users\Shubham\Desktop\SleepHealth-AI
python    : 3.13.2
pandas    : 3.0.6


In [2]:
import prepare_nhanes as pn

# Read and inner-join the six modules on the participant id SEQN.
# prepare_nhanes also repairs a read_sas quirk that decodes the XPT
# integer 0 as a denormalised float (5.4e-79).
merged = pn.load_merged()
print('merged shape:', merged.shape)

  P_DEMO.XPT   (15560, 29)
  P_BMX.XPT    (14300, 22)
  P_SLQ.XPT    (10195, 11)
  P_DIQ.XPT    (14986, 28)
  P_BPQ.XPT    (10195, 11)
  P_MCQ.XPT    (14986, 63)
merged shape: (9445, 159)


In [3]:
# Adult cohort: age >= 20 and valid diabetes / blood-pressure answers.
cohort = merged[(merged['RIDAGEYR'] >= 20)
                & merged['DIQ010'].isin([1, 2])
                & merged['BPQ020'].isin([1, 2])].copy()
print('adult cohort:', cohort.shape[0], 'participants')
print('age range   :', int(cohort['RIDAGEYR'].min()), '-', int(cohort['RIDAGEYR'].max()))

adult cohort: 8282 participants
age range   : 20 - 80


In [4]:
# Rebuild the feature + label table exactly as src/prepare_nhanes.py does.
# CDC codes 7 (Refused) and 9 (Don't know) are treated as missing, not
# as numeric values - leaving them in would corrupt the ordinal scales.
out = pd.DataFrame(index=cohort.index)
out['age'] = cohort['RIDAGEYR']
out['gender_female'] = (cohort['RIAGENDR'] == 2).astype(int)
out['bmi'] = cohort['BMXBMI']
out['waist_circumference'] = cohort['BMXWAIST']
out['sleep_duration_hrs'] = cohort['SLD012']          # weekday
out['weekend_sleep_hrs'] = cohort['SLD013']           # weekend
out['social_jetlag_hrs'] = (cohort['SLD013'] - cohort['SLD012']).round(2)

def freq(col, max_code):
    s = cohort[col].replace(pn.INVALID, np.nan)
    return s.where(s <= max_code).astype(float)

out['snore_freq'] = freq('SLQ030', 3)
out['witnessed_apnea_freq'] = freq('SLQ040', 3)
out['daytime_sleepiness'] = freq('SLQ120', 4)
out['trouble_sleeping'] = cohort['SLQ050'].replace(pn.INVALID, np.nan).map({1.0: 1, 2.0: 0})

In [5]:
# ---- The five prediction targets ---------------------------------------
heart_cols = ['MCQ160B', 'MCQ160C', 'MCQ160D', 'MCQ160E']
heart = cohort[heart_cols].replace(pn.INVALID, np.nan)
answered = heart.notna().any(axis=1)
out['target_heart_disease'] = ((heart == 1).any(axis=1)).astype(int).where(answered, np.nan)
out['target_insomnia_risk'] = out['trouble_sleeping']
osa = ((out['snore_freq'] == 3) | (out['witnessed_apnea_freq'] >= 2)).astype(int)
out['target_osa_risk'] = osa.where(out['snore_freq'].notna()
                                   & out['witnessed_apnea_freq'].notna())
out['target_diabetes'] = (cohort['DIQ010'] == 1).astype(int)
out['target_hypertension'] = (cohort['BPQ020'] == 1).astype(int)

print('final table:', out.shape)

final table: (8282, 16)


In [6]:
# ---- Feature / label summary ------------------------------------------
profile = pd.DataFrame({
    'dtype': out.dtypes.astype(str),
    'missing': out.isna().sum(),
    'missing_%': (out.isna().mean() * 100).round(1),
})
num = out.select_dtypes(include=[np.number])
profile['min'] = num.min()
profile['mean'] = num.mean().round(2)
profile['max'] = num.max()
profile

,dtype,missing,missing_%,min,mean,max
age,float64,0,0.0,20.0,50.77,80.0
gender_female,int64,0,0.0,0.0,0.52,1.0
bmi,float64,159,1.9,14.2,29.99,92.3
waist_circumference,float64,474,5.7,57.9,100.86,178.0
sleep_duration_hrs,float64,76,0.9,2.0,7.58,14.0
weekend_sleep_hrs,float64,81,1.0,2.0,8.26,14.0
social_jetlag_hrs,float64,105,1.3,-9.0,0.68,10.0
snore_freq,float64,597,7.2,0.0,1.52,3.0
witnessed_apnea_freq,float64,451,5.4,0.0,0.42,3.0
daytime_sleepiness,float64,16,0.2,0.0,1.76,4.0


In [7]:
# ---- Label prevalence ---------------------------------------------------
labels = [c for c in out.columns if c.startswith('target_')]
prev = pd.DataFrame({
    'positives': [int(out[c].sum()) for c in labels],
    'n': [int(out[c].notna().sum()) for c in labels],
})
prev['prevalence_%'] = (prev['positives'] / prev['n'] * 100).round(2)
prev.index = labels
prev

,positives,n,prevalence_%
target_heart_disease,774,8281,9.35
target_insomnia_risk,2371,8275,28.65
target_osa_risk,2391,7444,32.12
target_diabetes,1318,8282,15.91
target_hypertension,3164,8282,38.20


## Reproducibility check

The cell below compares what this notebook just rebuilt against the CSV the production pipeline ships (`data/processed/nhanes_sleep_health.csv`). They must be identical - if they diverge, this cell raises.

In [8]:
shipped = pd.read_csv('data/processed/nhanes_sleep_health.csv')

assert out.shape == shipped.shape, (out.shape, shipped.shape)
for col in shipped.columns:
    a, b = out[col].reset_index(drop=True), shipped[col]
    if a.dtype.kind == 'f' or b.dtype.kind == 'f':
        same = np.allclose(a.astype(float), b.astype(float), equal_nan=True)
    else:
        same = a.equals(b.astype(a.dtype))
    assert same, f'column mismatch: {col}'
print(f'OK - notebook rebuild matches shipped CSV exactly  ({len(shipped)} rows x {len(shipped.columns)} cols)')

OK - notebook rebuild matches shipped CSV exactly  (8282 rows x 16 cols)


In [9]:
# Cross-check cohort size against the metrics the app reports.
metrics = json.load(open('results/metrics/metrics.json', encoding='utf-8'))
assert len(shipped) == metrics['data']['n_participants'], 'cohort size drifted'
print('OK - cohort n =', len(shipped), 'matches metrics.json')

OK - cohort n = 8282 matches metrics.json
